# From resonate-and-fire neurons to a wave sheet

A step-by-step construction of `PhasorWaveSheet`. We start from a single
resonate-and-fire (R&F) neuron and add **one modification at a time** until we
have a 2-D sheet that supports self-sustaining traveling waves. Each piece is
built by hand first, then we show the package function that encapsulates it.

The three modifications, in order:

1. **Spatial coupling** — a difference-of-Gaussians (Mexican-hat) kernel so neurons talk to their neighbors.
2. **Conduction delays as phase factors** — a delay `τ` becomes a rotation `e^{-iωτ}` on the shared carrier, keeping everything a linear complex system (no delay-ODE).
3. **Phase-only saturation** — project each neuron onto the unit circle every step, which makes the waves self-limiting.

Plus the analysis tools that come for free: the **dispersion relation** and a **criticality** knob.

See also `src/wave.jl` and `docs/rf_wave_network_implementation.md`.

In [ ]:
using PhasorNetworks, Lux, Plots, FFTW, Random, Statistics
default(size=(560, 340), legend=:topright)

# Shared dynamics for the whole notebook.
λ = -0.15f0          # subthreshold damping (decay rate); Re of the eigenvalue
ω = Float32(2π)      # shared carrier frequency (per-channel-ω rule): every neuron rotates at ω
k = λ + im*ω         # the R&F eigenvalue k = λ + iω
T = 1.0f0            # one oscillation period (the discrete sample interval)
println("eigenvalue k = ", k, "    per-period decay |exp(kT)| = ", abs(exp(k*T)))

# helper: the gain g that puts a sheet exactly at criticality (spectral radius = 1),
# by geometric bisection on any function srf(g) -> spectral radius.
function critical_gain(srf; lo=1f-3, hi=1f2, iters=40)
    for _ in 1:iters
        m = sqrt(lo*hi); srf(m) < 1f0 ? (lo = m) : (hi = m)
    end
    return sqrt(lo*hi)
end

## 1. The resonate-and-fire neuron — one complex oscillator

A single R&F neuron is a damped complex oscillator:

$$\frac{dz}{dt} = (\lambda + i\omega)\,z = k\,z, \qquad z(t) = z(0)\,e^{kt}.$$

The complex state `z` bundles two real quantities: `Re z` is current-like and
`Im z` is voltage-like. A **spike** is a threshold crossing of the voltage
`Im z`. With `λ < 0` the trajectory spirals inward (damped subthreshold
oscillation); `ω` sets how fast it rotates.

This is *exactly* the per-channel dynamics the whole SSM codebase is built on —
`k = lambda + i*omega`.

In [ ]:
ts = range(0f0, 8f0, length=800)
z = exp.(k .* ts)                          # z(0) = 1

p_spiral = plot(real.(z), imag.(z); aspect_ratio=1, label="z(t)",
    title="single R&F neuron: a damped spiral",
    xlabel="Re z  (current-like)", ylabel="Im z  (voltage-like)")
scatter!(p_spiral, [1f0], [0f0]; label="z(0)")

p_time = plot(ts, imag.(z); lw=2, label="Im z  (voltage)", xlabel="t",
    title="ring-down over time")
plot!(p_time, ts, real.(z); lw=2, label="Re z  (current)")
plot(p_spiral, p_time, layout=(1,2), size=(900,340))

## 2. Discrete time — the phasor-SSM view

Sampling the ODE once per period `Δt = T` gives an exact linear recurrence

$$z[n{+}1] = A\,z[n] + B\,I[n], \qquad A = e^{kT},\; B = \tfrac{A-1}{k}.$$

`A` is the per-period propagator; `|A| = e^{\lambda T} < 1` is the decay. This
is the discrete "kernel" view (`phasor_kernel`, `causal_conv`) that lets the
network train efficiently. Everything below builds on this single-step update.

In [ ]:
A = exp(k*T)                      # per-period propagator
zn = ComplexF32[1]
for n in 1:6; push!(zn, A*zn[end]); end
println("A = ", A, "   |A| = ", abs(A))
plot(0:6, abs.(zn); marker=:o, lw=2, label="|z[n]| (free decay)",
     xlabel="period n", ylabel="|z|", title="discrete R&F: z[n+1] = A z[n]")

## 3. Many neurons on a 2-D sheet — but no coupling yet

Now place an `S×S` grid of R&F neurons. Per the **per-channel-ω rule**, every
neuron shares the *same* carrier `ω` (so their phases stay comparable for
downstream VSA operations); diversity will come from `λ` and the coupling, not
from `ω`.

Without any coupling, each neuron evolves independently: `z[t+1] = A·z[t]`. A
localized pulse just **decays in place** — there is no wave. Coupling is what we
add next.

In [ ]:
S = 40
cen = S ÷ 2
z0 = zeros(ComplexF32, S, S); z0[cen-1:cen+1, cen-1:cen+1] .= 1f0
snaps = accumulate((zz, _) -> A .* zz, 1:8; init=z0)     # snaps[t] = state after t steps
plot(heatmap(abs.(z0); title="t = 0"),
     heatmap(abs.(snaps[8]); title="t = 8  (just decays, no spread)"),
     layout=(1,2), aspect_ratio=1, c=:viridis, size=(820,330), clims=(0,1))

## 4. Modification 1 — spatial coupling (difference-of-Gaussians)

Give each neuron input from its neighbors, weighted by distance:

$$\frac{dz_i}{dt} = k\,z_i + g\sum_j W(\lVert r_i - r_j\rVert)\,z_j.$$

`W(r)` is a **difference-of-Gaussians** (Mexican hat): local excitation (narrow
positive) minus a broader inhibitory surround (`σ_I > σ_E`). Tuned near
**spatial balance** `A_exc·σ_E² ≈ B_inh·σ_I²` (so `∫W ≈ 0`), it selects a
preferred wavelength. `g` is a global gain (our criticality knob, §9). The self
term (`r = 0`) is zeroed — self-dynamics already live in `A`.

In [ ]:
# toroidal (wrap-around) distance from the origin (1,1) — the right convention
# for a circular (FFT) convolution kernel centered at the origin
function wrapped_r(S)
    r = zeros(Float32, S, S)
    for j in 1:S, i in 1:S
        di = i-1; di = di > S÷2 ? di-S : di
        dj = j-1; dj = dj > S÷2 ? dj-S : dj
        r[i,j] = sqrt(Float32(di)^2 + Float32(dj)^2)
    end
    r
end
rg = wrapped_r(S)
Aexc, σe, Binh, σi = 1f0, 1.5f0, 0.25f0, 3.0f0
mag = Aexc .* exp.(-rg.^2 ./ (2σe^2)) .- Binh .* exp.(-rg.^2 ./ (2σi^2))
mag = mag .* (rg .> 0f0)                 # zero self-coupling
heatmap(fftshift(mag); aspect_ratio=1, c=:balance, title="W(r): difference-of-Gaussians (Mexican hat)",
        xlabel="Δx", ylabel="Δy")

## 5. Modification 2 — conduction delays as phase factors

Signals take time to travel: a delay `τ(r) = r/c` for conduction speed `c`.
Normally delays force a delay-differential equation. But because the whole sheet
is **phase-locked** to one carrier `ω`, a delay is *exactly* a phase rotation:

$$s(t-\tau)\;\longrightarrow\; e^{-i\omega\tau}\,s(t).$$

So the delay just makes the coupling weights **complex** — we stay inside a
linear complex system, no DDE solver. This is the single biggest simplification,
and it only works thanks to the shared-ω discipline.

In [ ]:
c = 40f0                                        # conduction speed (pixels per period)
delay = exp.(-im .* ω .* rg ./ c)              # τ = r/c  →  phase e^{-iωτ}
Wfull = ComplexF32.(mag) .* delay              # complex coupling kernel
plot(heatmap(fftshift(real.(Wfull)); title="Re W", c=:balance, aspect_ratio=1),
     heatmap(fftshift(imag.(Wfull)); title="Im W  (introduced by delays)", c=:balance, aspect_ratio=1),
     layout=(1,2), size=(820,330))

## 6. Why it's fast — the spatial FFT

The coupling `Σ_j W(r_i - r_j) z_j` is a **convolution** (it depends only on
`r_i - r_j`). Translation-invariant convolutions **diagonalize under the FFT**,
so the whole coupling collapses to a pointwise multiply in the spatial-frequency
domain:

$$\text{coupled} = \mathrm{ifft2}\big(\hat W \odot \mathrm{fft2}(z)\big), \qquad \hat W = \mathrm{fft2}(W).$$

This is `O(S^2 \log S)`, GPU-native (cuFFT), and differentiable. `Ŵ(q)` is the
coupling's response at each spatial frequency `q` — we'll read the dynamics
straight off it in §9.

In [ ]:
What = fft(Wfull)
heatmap(fftshift(abs.(What)); aspect_ratio=1, c=:viridis,
        title="|Ŵ(q)| — coupling response per spatial frequency q",
        xlabel="qₓ", ylabel="q_y")

## 7. Put it together — a traveling wave (by hand)

One recurrent step is now:

$$z[t{+}1] = A\,z[t] \;+\; g\,\mathrm{ifft2}(\hat W \odot \mathrm{fft2}(z[t])) \;+\; \text{drive}.$$

We pick the gain `g` near **criticality** (so a disturbance neither dies
instantly nor blows up — see §9), seed a point pulse, and watch it propagate
outward as a wave.

In [ ]:
# choose g near criticality: the per-step multiplier is M(q) = A + g Ŵ(q);
# criticality is where its largest magnitude (spectral radius) reaches 1.
gs = range(1f-3, 0.1f0, length=600)
srs = [maximum(abs.(A .+ gg .* What)) for gg in gs]
gcrit_manual = gs[findfirst(srs .>= 1f0)]
g = 0.98f0 * gcrit_manual
println("manual g_crit ≈ ", round(gcrit_manual, digits=4), "   using g = ", round(g, digits=4))

wave_step(zz) = A .* zz .+ g .* ifft(What .* fft(zz))
z0 = zeros(ComplexF32, S, S); z0[cen, cen] = 1f0            # seed a point pulse
L = 24
snaps = accumulate((zz, _) -> wave_step(zz), 1:L; init=z0)  # snaps[t] = state after t steps
tvals = (0, 6, 12, 18, 24)
frames = [z0, snaps[6], snaps[12], snaps[18], snaps[24]]
plot([heatmap(abs.(frames[i]); title="t=$(tvals[i])", axis=false, colorbar=false, aspect_ratio=1)
      for i in 1:5]...,
     layout=(1,5), size=(1050,230), c=:viridis)

## 8. The package layer does exactly this

`PhasorWaveSheet` bundles the eigenvalue, the DoG-with-delays kernel, the FFT
coupling, and the recurrence — with all the pieces as trainable parameters.
`wave_simulate` runs the autonomous rollout. Built with the same constants, it
reproduces our hand-rolled wave (up to the first-step bookkeeping).

In [ ]:
sheet = PhasorWaveSheet(S, S; saturating=false,
    init_log_neg_lambda=log(0.15), init_A_exc=1.0, init_log_sigma_exc=log(1.5),
    init_B_inh=0.25, init_log_sigma_inh=log(3.0), init_log_speed=log(40.0),
    init_log_g=log(g))
ps, st = Lux.setup(Xoshiro(0), sheet)
z0 = zeros(ComplexF32, S, S); z0[cen, cen] = 1f0
traj = wave_simulate(sheet, ps, st; z0=z0, L=L)          # (S, S, L)

# package traj[:,:,t] corresponds to our manual snaps[t]
println("max |manual - package| at final step = ",
        maximum(abs.(traj[:, :, L] .- snaps[L])))
plot([heatmap(abs.(traj[:,:,t]); title="t=$t", axis=false, colorbar=false, aspect_ratio=1)
      for t in (1, 7, 13, 19, 24)]...,
     layout=(1,5), size=(1050,230), c=:viridis)

## 9. Dispersion & criticality — for free

Because the coupling is FFT-diagonal, each spatial frequency `q` evolves
*independently* with per-step multiplier

$$M(q) = A + g\,\hat W(q).$$

- `|M(q)|` is the per-step growth of mode `q`. The **spectral radius**
  `max_q |M(q)|` is the phase-SSM analogue of the branching ratio σ: `<1`
  extinguishes, `≈1` self-sustaining (critical), `>1` saturates.
- `arg M(q)` gives the per-step phase advance ⇒ wave speed.

`dispersion(sheet, ps, st)` returns all of this. The growth-rate map shows
*which* wavelengths grow — for a balanced Mexican hat, a ring at nonzero `|q|`
(a preferred wavelength), not the DC (uniform) mode.

In [ ]:
d = dispersion(sheet, ps, st)
println("spectral radius = ", round(d.spectral_radius, digits=4))

# gain that puts the sheet exactly at criticality
setg(gv) = merge(ps, (log_g = Float32[log(gv)],))
srf(gv)  = dispersion(sheet, setg(gv), st).spectral_radius
gcrit = critical_gain(srf)
println("g_crit (bisection) = ", round(gcrit, digits=4))

heatmap(fftshift(d.growth_rate); aspect_ratio=1, c=:viridis,
        title="growth rate Re k_eff(q): which spatial-frequency waves grow",
        xlabel="qₓ", ylabel="q_y")

## 10. Modification 3 — self-limiting via phase-only saturation

A *linear* wave sheet can't regulate itself: above criticality its amplitude
grows geometrically. Because phasor networks carry information in **phase**, we
can project every neuron back onto the unit circle each step
(`normalize_to_unit_circle`, the `saturating=true` option):

$$z[t] \leftarrow z[t] / |z[t]|.$$

This bounds `|z| ≡ 1` for free — no threshold/reset needed. Below, at a
supercritical gain, the linear sheet diverges while the saturating sheet stays
bounded.

> ⚠️ **Important consequence for readout:** phase-only saturation *throws away
> magnitude*. Any information a task encodes in wave **amplitude/intensity**
> (e.g. interference strength) is erased by saturation — see §11.

In [ ]:
gsuper = log(1.6f0 * gcrit)
mk(sat) = PhasorWaveSheet(S, S; saturating=sat,
    init_log_neg_lambda=log(0.15), init_A_exc=1.0, init_log_sigma_exc=log(1.5),
    init_B_inh=0.25, init_log_sigma_inh=log(3.0), init_log_speed=log(40.0), init_log_g=gsuper)
lin = mk(false); sat = mk(true)
pl, sl = Lux.setup(Xoshiro(0), lin); psat, ssat = Lux.setup(Xoshiro(0), sat)
z0 = zeros(ComplexF32, S, S); z0[cen, cen] = 1f0
tl  = wave_simulate(lin,  pl,  sl;  z0=z0, L=40)
tsa = wave_simulate(sat, psat, ssat; z0=z0, L=40)
plot(1:40, [maximum(abs.(tl[:,:,t]))  for t in 1:40]; yscale=:log10, lw=2, label="linear (grows)",
     xlabel="step", ylabel="max |z|  (log)", title="phase-only saturation self-limits amplitude")
plot!(1:40, [maximum(abs.(tsa[:,:,t])) for t in 1:40]; lw=2, label="saturating (|z| ≡ 1)")

## 11. Reading the sheet — where does the information live?

To *use* the sheet we read some of its cells over time. A clean example of what
the wave computes: put **two point sources** near the center with a phase
difference `Δφ`, drive them continuously, and read the **outer perimeter**.

The two waves interfere. The perimeter **intensity** `|z|²` pattern depends on
`Δφ` — a genuinely nonlinear function of the inputs (it involves `cos Δφ`, a
product term). Below, `Δφ = 0` (constructive) and `Δφ = π` (destructive) give
clearly different perimeter intensity patterns.

Two lessons this teaches about readout:
- The nonlinear relation is **in the magnitude** (intensity), not linearly in
  `(Re z, Im z)`.
- So a *phase-only saturating* sheet (§10) would **erase** it, and a purely
  linear map on `(Re z, Im z)` can't recover it (it can't square). Extracting it
  needs an intensity (`|z|²`) read — exactly the square-law an optical detector
  performs. This is why the classifier experiments behaved the way they did.

In [ ]:
S2 = 24; cen2 = S2 ÷ 2; dpos = 5
sheet2 = PhasorWaveSheet(S2, S2; saturating=false,
    init_log_neg_lambda=log(0.15), init_A_exc=1.0, init_log_sigma_exc=log(1.5),
    init_B_inh=0.25, init_log_sigma_inh=log(3.0), init_log_speed=log(40.0), init_log_g=log(0.02))
p2, s2 = Lux.setup(Xoshiro(0), sheet2)
# put this sheet near criticality
setg2(gv) = merge(p2, (log_g = Float32[log(gv)],))
srf2(gv)  = dispersion(sheet2, setg2(gv), s2).spectral_radius
p2 = setg2(0.9f0 * critical_gain(srf2))

perim = [i + (j-1)*S2 for j in 1:S2 for i in 1:S2 if (i==1 || i==S2 || j==1 || j==S2)]
function perimeter_intensity(Δφ)
    L2 = 20
    drive = zeros(ComplexF32, S2, S2, L2, 1)
    drive[cen2, cen2-dpos, :, 1] .= 1f0                       # source A, phase 0
    drive[cen2, cen2+dpos, :, 1] .= exp(im * Float32(π) * Δφ) # source B, phase Δφ·π
    tr = wave_simulate(sheet2, p2, s2; z0=zeros(ComplexF32, S2, S2, 1), L=L2, drive=drive)
    per = reshape(tr, S2*S2, L2, 1)[perim, end, 1]
    abs2.(per)                                                # perimeter intensity |z|²
end
plot(perimeter_intensity(0f0);  lw=2, label="Δφ = 0  (same)",
     title="perimeter interference intensity |z|² encodes Δφ",
     xlabel="perimeter position", ylabel="|z|²")
plot!(perimeter_intensity(1f0); lw=2, label="Δφ = π  (opposite)")

## Summary — the three modifications

Starting from the R&F neuron `dz/dt = (λ+iω)z`, the wave sheet adds:

1. **Spatial DoG coupling** (§4) — neurons drive their neighbors via a Mexican-hat kernel → waves can form.
2. **Conduction delays as phase factors** `e^{-iωτ}` (§5) + **FFT diagonalization** (§6) → waves *travel*, cheaply and differentiably.
3. **Phase-only saturation** (§10) → waves *self-limit* (and, as a side effect, discard amplitude).

The **dispersion relation** `M(q) = A + g·Ŵ(q)` (§9) hands us wave speed and a
one-scalar **criticality** knob `g` (spectral radius ≈ 1).

Two modes of the *same* equation live on one struct: the discrete recurrence we
built here (Tier 1, trainable) and a continuous ODE (Tier 2, `CurrentCall`) —
they agree to field-similarity ≈ 0.999. And §11 is the key readout caveat: the
sheet's computation often lives in wave **intensity**, which a phase-only sheet
erases and a strictly-linear `(Re, Im)` readout can't see.

Next: `src/wave.jl` (implementation), `docs/rf_wave_network_implementation.md`
(design + results), `demos/wave_dispersion.jl` (dispersion/ablation demo).